cell install

In [ ]:
%pip install networkx openai

cell 2 Import

In [ ]:
import xml.etree.ElementTree as ET
import networkx as nx
import re
from openai import AzureOpenAI
import os

cell 3 XML files Path ( 6 files)

In [ ]:
xml_files = [
    "/dbfs/FileStore/xml_data/file1.xml",
    "/dbfs/FileStore/xml_data/file2.xml",
    "/dbfs/FileStore/xml_data/file3.xml",
    "/dbfs/FileStore/xml_data/file4.xml",
    "/dbfs/FileStore/xml_data/file5.xml",
    "/dbfs/FileStore/xml_data/file6.xml"
]

print("Files Loaded:", xml_files)

CELL 4: CLEAN FUNCTION (IMPROVED)

def clean_name(name):
    if not name:
        return ""
    
    name = name.strip()
    name = re.sub(r'[^\w\s]', '', name)  # remove special chars
    return name.title()

CELL 5: PARSE XML (🔥 FINAL VERSION)

In [ ]:
def parse_xml_files(xml_files):
    
    data = []

    for file_path in xml_files:

        print(f"\nProcessing: {file_path}")

        try:
            tree = ET.parse(file_path)
            root = tree.getroot()

            for page in root.findall(".//page"):

                # Title
                title_elem = page.find("title")
                title = clean_name(title_elem.text) if title_elem is not None else None

                # Text
                text_elem = page.find("text")
                text = text_elem.text if text_elem is not None else ""

                # Extract LINKS
                links = []

                # <link> tags
                for link in page.findall(".//link"):
                    if link.text:
                        links.append(clean_name(link.text))

                # [[...]] pattern
                if text:
                    matches = re.findall(r"\[\[(.*?)\]\]", text)
                    links.extend([clean_name(m) for m in matches])

                # 🔥 Remove noise + duplicates
                links = list(set([
                    l for l in links
                    if l and len(l) > 2 and l != title
                ]))

                data.append({
                    "doc_id": title,
                    "text": text,
                    "links": links
                })

        except Exception as e:
            print(f"Error in {file_path}: {e}")

    return data


data = parse_xml_files(xml_files)

print("\nTotal Pages:", len(data))

CELL 6: BUILD GRAPH (GLOBAL GRAPH 🔥)

graph = nx.DiGraph()

for row in data:
    source = row["doc_id"]

    if not source:
        continue

    # add node
    graph.add_node(source, text=row["text"])

    # add edges
    for target in row["links"]:
        graph.add_edge(source, target)

print("Nodes:", len(graph.nodes()))
print("Edges:", len(graph.edges()))

CELL 7: PageRank (Importance)

pagerank_scores = nx.pagerank(graph)

for node, score in pagerank_scores.items():
    graph.nodes[node]["pagerank"] = score

print("\nTop Important Nodes:")
top_nodes = sorted(pagerank_scores.items(), key=lambda x: x[1], reverse=True)[:10]

for node, score in top_nodes:
    print(node, ":", round(score, 4))

CELL 8: GRAPH RETRIEVER

def graph_retriever(graph, query, max_hops=2):

    query_words = query.lower().split()
    context = []
    matched_nodes = []

    # 🔥 Better matching
    for node in graph.nodes():
        node_lower = node.lower()

        if any(word in node_lower for word in query_words):
            matched_nodes.append(node)

    print("Matched Nodes:", matched_nodes)

    # 🔥 Multi-hop traversal
    for node in matched_nodes:

        paths = nx.single_source_shortest_path_length(
            graph, node, cutoff=max_hops
        )

        for target in paths:

            # add node text
            text = graph.nodes[target].get("text", "")
            if text:
                context.append(text[:200])

            # add relationships
            for neighbor in graph.neighbors(target):
                context.append(f"{target} → {neighbor}")

    return "\n".join(context[:25])

CELL 9: Azure OpenAI Setup

client = AzureOpenAI(
    api_key=os.getenv("AZURE_OPENAI_API_KEY"),
    azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT"),
    api_version="2024-02-15-preview"
)

DEPLOYMENT_NAME = os.getenv("AZURE_OPENAI_DEPLOYMENT")

CELL 10: FINAL Graph RAG

def graph_rag_answer(graph, query):

    context = graph_retriever(graph, query)

    print("\n🔍 Retrieved Context:\n")
    print(context)

    prompt = f"""
You are an expert system.

Use ONLY the context below:

{context}

Question: {query}

Answer clearly and concisely:
"""

    response = client.chat.completions.create(
        model=DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": "You are helpful."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.3
    )

    return response.choices[0].message.content

CELL 11: TEST

queries = [
    "How combustion works?",
    "How fuel system works with combustion?",
    "How engine connects to turbine?",
    "Explain full system flow"
]

for q in queries:
    print("\n====================")
    print("Query:", q)

    answer = graph_rag_answer(graph, q)

    print("\n🤖 Answer:\n", answer)

6 XML files
   ↓
Extract pages + multiple links
   ↓
Global graph (multi-reference)
   ↓
Multi-hop traversal
   ↓
Context
   ↓
LLM
   ↓
Answer